# Tâche 6: Normalisation du texte

**Projet 1 · African Language Data & LLM Readiness**

**Consigne** : normaliser le texte en préservant les spécificités linguistiques (caractères spécifiques, diacritiques, ponctuation, formatage significatif).

Le code est dans `src/normaliser.py`. Il lit le corpus de la collecte (tâche 5, dossier indiqué dans `config/config.yaml`, section `normalisation`), sans jamais le modifier, et écrit le résultat dans `data/processed/normalise/<run>/`.

| Règle | Ce qu'elle fait |
|---|---|
| R1 | Unicode NFC (les tons ajoutés aux lettres sont gardés) |
| R2 | espaces spéciales (insécable, fine…) → espace normale |
| R3 | caractères invisibles inutiles retirés (ZWJ et ZWNJ gardés) |
| R4 | caractères de contrôle réparés (`\x93…\x94` → “…”) ou retirés |
| R5 | sosies d'un autre alphabet dans un mot latin → bonne lettre (`ε` → `ɛ`, `ↄ` → `ɔ`, `Ð` → `Ɖ`…) |
| R6 | ponctuation détournée (fon : `‚` → `,`) |
| R7 | restes du nettoyage : crochets orphelins, « File:\| », parenthèses vides |
| R8 | espaces et lignes vides en trop |
| R9 | apostrophes et guillemets uniformisés : `’` → `'`, `“ ”` → `"`, `« texte »` → `"texte"`, `‹ ›` → `'` (`ʼ` est gardé) |

Jamais modifiés : lettres africaines, tons et diacritiques, écritures non latines, mots cités dans un autre alphabet, casse, chiffres.

| Commande | Effet |
|---|---|
| `python src/normaliser.py --langues tw ee fon` | quelques langues |
| `python src/normaliser.py --tout` | toutes les langues du corpus |
| `python src/normaliser.py --reprendre` | reprend un run interrompu (les langues terminées ne sont pas refaites) |

In [ ]:
%pip install -q -r ../requirements.txt

In [7]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NORMALISE = ROOT / "data" / "processed" / "normalise"


def lancer(*args):
    # Lance src/normaliser.py et affiche son journal au fur et à mesure.
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.Popen([sys.executable, "-u", str(ROOT / "src" / "normaliser.py"), *args], cwd=ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        p.terminate()
        print("\nRun arrêté. Pour continuer : lancer('--reprendre')")
    return p.wait()


def dernier_run():
    return sorted(NORMALISE.glob("*T*Z"))[-1]


def resultats(run=None):
    run = Path(run) if run else dernier_run()
    etat = json.loads((run / "etat.json").read_text(encoding="utf-8"))
    print("Run", run.name, ":", etat["statut"])
    for nom, ok in (etat.get("controles") or {}).items():
        print(f"  {nom} : {'OK' if ok else 'NON'}")
    cc = etat.get("controle_complet")
    if cc:
        print(f"  contrôle complet ({cc['textes_controles']} textes, {cc['textes_en_erreur']} en erreur) : {'OK' if cc['ok'] else 'NON'}")
    else:
        print("  contrôle complet : pas encore fait (étape 4)")
    lignes = []
    for p in sorted((run / "mesures").glob("*.json")):
        m = json.loads(p.read_text(encoding="utf-8"))
        lignes.append({"langue": m["langue"], "textes avant": m["textes_entree"], "textes après": m["textes_sortie"],
                       "textes modifiés": m["textes_modifies"], "écartés": sum(m["ecartes"].values()),
                       **{r.split()[0]: n for r, n in m["regles"].items()},
                       "contrôles": "OK" if not m["controles_normalisation"] and not m["erreurs_validation"] else "NON"})
    return pd.DataFrame(lignes).set_index("langue").fillna(0) if lignes else pd.DataFrame()


def exemples(code, run=None, par_regle=3):
    # Affiche quelques exemples avant / après pour chaque règle appliquée à une langue.
    run = Path(run) if run else dernier_run()
    m = json.loads((run / "mesures" / f"{code}.json").read_text(encoding="utf-8"))
    for regle, ex in m["exemples"].items():
        print(f"\n{regle} ({m['regles'].get(regle, 0)} fois)")
        for e in ex[:par_regle]:
            print("   avant :", repr(e["avant"]))
            print("   après :", repr(e["apres"]))

## Étape 1: Test sur quelques langues

Wolof, twi, éwé, fon, kabyle, n'ko et amharique : ces corpus contiennent tous les cas visés par les règles (sosies en twi et en kabyle, ponctuation détournée en fon, tons du fon et du n'ko, écriture guèze…). Quelques secondes.

In [2]:
lancer("--langues", "wo", "tw", "ee", "fon", "kab", "nqo", "am", "--controle-complet")
resultats()

2026-11-13T14:12:02 INFO Run 20261113T141202Z : 7 langue(s) à normaliser (0 déjà faites), source data\processed\collecte\20261105T065025Z, 15 processus
2026-11-13T14:12:06 INFO ee : 1723 textes -> 1723, 433 modifiés, 0 écartés, contrôles OK (2 s)
2026-11-13T14:12:06 INFO wo : 1809 textes -> 1809, 270 modifiés, 0 écartés, contrôles OK (2 s)
2026-11-13T14:12:08 INFO nqo : 1726 textes -> 1726, 478 modifiés, 0 écartés, contrôles OK (4 s)
2026-11-13T14:12:08 INFO fon : 6177 textes -> 6177, 3813 modifiés, 0 écartés, contrôles OK (4 s)
2026-11-13T14:12:10 INFO kab : 7315 textes -> 7315, 1652 modifiés, 0 écartés, contrôles OK (6 s)
2026-11-13T14:12:13 INFO tw : 5642 textes -> 5641, 2592 modifiés, 1 écartés, contrôles OK (9 s)
2026-11-13T14:12:14 INFO am : 16286 textes -> 16286, 2538 modifiés, 0 écartés, contrôles OK (10 s)
2026-11-13T14:12:14 INFO Run 20261113T141202Z terminé : 40678 textes -> 40677
2026-11-13T14:12:14 INFO   corpus valide et 100 % des textes avec provenance : OK
2026-11-13T14

,textes avant,textes après,textes modifiés,écartés,R2,R3,R4,R5,R7,R8,R9,contrôles,R6,R1
langue,,,,,,,,,,,,,,
am,16286,16286,2538,0,1669,1019,0,1.0,179,139,24314,OK,0.0,0.0
ee,1723,1723,433,0,485,128,0,756.0,20,30,526,OK,0.0,0.0
fon,6177,6177,3813,0,450,49,0,546.0,29,209,1456,OK,3913.0,0.0
kab,7315,7315,1652,0,3782,3764,4,775.0,106,1021,3107,OK,0.0,0.0
nqo,1726,1726,478,0,1935,61,0,0.0,35,277,1575,OK,0.0,0.0
tw,5642,5641,2592,1,2226,612,0,6036.0,192,395,5698,OK,0.0,1.0
wo,1809,1809,270,0,527,15,0,1.0,15,38,2846,OK,0.0,0.0


In [3]:
exemples("tw")


R2 espaces spéciales (2226 fois)
   avant : '\nJane Naana Opoku Agyeman\xa0(awoda Obubuo 22, 1951 O'
   après : '\nJane Naana Opoku Agyeman (awoda Obubuo 22, 1951 O'
   avant : 'musini a ɔwɔ Switzerland.\xa0 Wɔ woo Liman a n’abusua'
   après : 'musini a ɔwɔ Switzerland.  Wɔ woo Liman a n’abusua'
   avant : 'fre won kwawuh foo, 11.41\xa0km².\n\nDin\nEdin Uetersen '
   après : 'fre won kwawuh foo, 11.41 km².\n\nDin\nEdin Uetersen '

R5 sosies (6036 fois)
   avant : 'o 22, 1951 Oguaa Gaana) yε Gaana Nwomasua Soafoɔ a'
   après : 'o 22, 1951 Oguaa Gaana) yɛ Gaana Nwomasua Soafoɔ a'
   avant : 'ts Strength)\n\nƆbaa ene abε dua\n\n(The woman with th'
   après : 'ts Strength)\n\nƆbaa ene abɛ dua\n\n(The woman with th'
   avant : 'Yɛwo Anansesεm pii, enti wonim ebi a, twerε wo ha '
   après : 'Yɛwo Anansesɛm pii, enti wonim ebi a, twerɛ wo ha '

R9 apostrophes et guillemets (5698 fois)
   avant : " obi a n'ani mfiiɛ anaa n’adwene mu nnɔ no rente a"
   après : " obi a n'ani mfiiɛ anaa n'adwene

## Étape 2: Toutes les langues

Les langues sont traitées en parallèle, les plus gros corpus d'abord (arabe standard : 3,5 Go). S'il s'arrête, passer à l'étape 3.

Pendant le run, le contrôle « plus aucun caractère interdit, normalisation stable » ne porte que sur les 2 000 premiers textes de chaque langue. Le contrôle de chaque texte se fait ensuite à l'étape 4.

In [4]:
lancer("--tout")
resultats()

2026-11-13T14:14:39 INFO Run 20261113T141439Z : 69 langue(s) à normaliser (0 déjà faites), source data\processed\collecte\20261105T065025Z, 15 processus
2026-11-13T14:14:55 INFO zgh : 12192 textes -> 12192, 557 modifiés, 0 écartés, contrôles OK (13 s)
2026-11-13T14:15:03 INFO tn : 7309 textes -> 7309, 1259 modifiés, 0 écartés, contrôles OK (20 s)
2026-11-13T14:15:04 INFO ary : 20321 textes -> 20321, 556 modifiés, 0 écartés, contrôles OK (21 s)
2026-11-13T14:15:05 INFO so : 15572 textes -> 15572, 3434 modifiés, 0 écartés, contrôles OK (23 s)
2026-11-13T14:15:08 INFO am : 16286 textes -> 16286, 2538 modifiés, 0 écartés, contrôles OK (13 s)
2026-11-13T14:15:15 INFO dag : 15711 textes -> 15711, 2277 modifiés, 0 écartés, contrôles OK (33 s)
2026-11-13T14:15:17 INFO lg : 7109 textes -> 7109, 4517 modifiés, 0 écartés, contrôles OK (12 s)
2026-11-13T14:15:19 INFO rw : 12327 textes -> 12327, 7683 modifiés, 0 écartés, contrôles OK (15 s)
2026-11-13T14:15:20 INFO gpe : 5873 textes -> 5873, 1465 m

,textes avant,textes après,textes modifiés,écartés,contrôles,R1,R2,R3,R4,R5,R7,R8,R9,R6
langue,,,,,,,,,,,,,,
aa,0,0,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
af,131092,131092,36008,0,OK,2.0,118030.0,3837.0,45.0,28.0,368.0,1430.0,253148.0,0.0
ak,1,1,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
am,16286,16286,2538,0,OK,0.0,1669.0,1019.0,0.0,1.0,179.0,139.0,24314.0,0.0
ann,494,494,25,0,OK,0.0,16.0,10.0,0.0,0.0,8.0,8.0,13.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wo,1809,1809,270,0,OK,0.0,527.0,15.0,0.0,1.0,15.0,38.0,2846.0,0.0
xh,3090,3090,579,0,OK,0.0,1611.0,40.0,0.0,0.0,140.0,260.0,1050.0,0.0
yo,41803,41803,3403,0,OK,0.0,5051.0,1851.0,0.0,32.0,289.0,572.0,10907.0,0.0


## Étape 3: Reprendre un run interrompu

In [12]:
lancer("--reprendre")
resultats()

2026-11-13T14:03:27 INFO Run 20261113T135306Z : 7 langue(s) à normaliser (62 déjà faites), source data\processed\collecte\20261105T065025Z, 15 processus

Run arrêté. Pour continuer : lancer('--reprendre')
Run 20261113T135306Z : en cours


,textes avant,textes après,textes modifiés,écartés,contrôles,R2,R3,R4,R7,R9,R5,R8,R1,R6
langue,,,,,,,,,,,,,,
aa,0,0,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ak,1,1,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
am,16286,16286,2538,0,OK,1669.0,1019.0,0.0,179.0,24314.0,1.0,139.0,0.0,0.0
ann,494,494,25,0,OK,16.0,10.0,0.0,8.0,13.0,0.0,8.0,0.0,0.0
ary,20321,20321,556,0,OK,829.0,113.0,1.0,28.0,2210.0,0.0,132.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wo,1809,1809,270,0,OK,527.0,15.0,0.0,15.0,2846.0,1.0,38.0,0.0,0.0
xh,3090,3090,579,0,OK,1611.0,40.0,0.0,140.0,1050.0,0.0,260.0,0.0,0.0
yo,41803,41803,3403,0,OK,5051.0,1851.0,0.0,289.0,10907.0,32.0,572.0,0.0,0.0


## Étape 4: Contrôle complet de chaque texte

Relit chaque texte du dernier run terminé et vérifie qu'il est en NFC, sans caractère interdit, et qu'un 2e passage de la normalisation ne le change plus. Le corpus n'est pas modifié. Le résultat est écrit dans `controle_complet.json` et noté dans `etat.json`, `run_metadata.json` et `run.log`. Environ 20 minutes pour toutes les langues.

Pour un run précis : `lancer("--verifier", "20261113T141439Z")`.

In [8]:
lancer("--verifier")
resultats()

2026-11-13T15:25:13 INFO Contrôle complet du run 20261113T141439Z : 69 langue(s), 15 processus
2026-11-13T15:25:22 INFO zgh : 12192 textes contrôlés, 0 en erreur (5 s)
2026-11-13T15:25:26 INFO ary : 20321 textes contrôlés, 0 en erreur (9 s)
2026-11-13T15:25:26 INFO am : 16286 textes contrôlés, 0 en erreur (5 s)
2026-11-13T15:25:26 INFO tn : 7309 textes contrôlés, 0 en erreur (10 s)
2026-11-13T15:25:27 INFO so : 15572 textes contrôlés, 0 en erreur (10 s)
2026-11-13T15:25:32 INFO lg : 7109 textes contrôlés, 0 en erreur (5 s)
2026-11-13T15:25:32 INFO tum : 19880 textes contrôlés, 0 en erreur (5 s)
2026-11-13T15:25:33 INFO rw : 12327 textes contrôlés, 0 en erreur (7 s)
2026-11-13T15:25:33 INFO dag : 15711 textes contrôlés, 0 en erreur (17 s)
2026-11-13T15:25:33 INFO yo : 41803 textes contrôlés, 0 en erreur (17 s)
2026-11-13T15:25:34 INFO gpe : 5873 textes contrôlés, 0 en erreur (8 s)
2026-11-13T15:25:35 INFO nqo : 1726 textes contrôlés, 0 en erreur (2 s)
2026-11-13T15:25:36 INFO shi : 1095

,textes avant,textes après,textes modifiés,écartés,contrôles,R1,R2,R3,R4,R5,R7,R8,R9,R6
langue,,,,,,,,,,,,,,
aa,0,0,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
af,131092,131092,36008,0,OK,2.0,118030.0,3837.0,45.0,28.0,368.0,1430.0,253148.0,0.0
ak,1,1,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
am,16286,16286,2538,0,OK,0.0,1669.0,1019.0,0.0,1.0,179.0,139.0,24314.0,0.0
ann,494,494,25,0,OK,0.0,16.0,10.0,0.0,0.0,8.0,8.0,13.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wo,1809,1809,270,0,OK,0.0,527.0,15.0,0.0,1.0,15.0,38.0,2846.0,0.0
xh,3090,3090,579,0,OK,0.0,1611.0,40.0,0.0,0.0,140.0,260.0,1050.0,0.0
yo,41803,41803,3403,0,OK,0.0,5051.0,1851.0,0.0,32.0,289.0,572.0,10907.0,0.0


## Étape 5: Résultats

Tableau par langue (textes, nombre d'applications de chaque règle, contrôles) et exemples avant / après pour une langue.

In [5]:
resultats()

Run 20261113T141439Z : terminé
  corpus valide et 100 % des textes avec provenance : OK
  plus aucun caractère interdit, normalisation stable : OK
  toutes les langues traitées : OK


,textes avant,textes après,textes modifiés,écartés,contrôles,R1,R2,R3,R4,R5,R7,R8,R9,R6
langue,,,,,,,,,,,,,,
aa,0,0,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
af,131092,131092,36008,0,OK,2.0,118030.0,3837.0,45.0,28.0,368.0,1430.0,253148.0,0.0
ak,1,1,0,0,OK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
am,16286,16286,2538,0,OK,0.0,1669.0,1019.0,0.0,1.0,179.0,139.0,24314.0,0.0
ann,494,494,25,0,OK,0.0,16.0,10.0,0.0,0.0,8.0,8.0,13.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wo,1809,1809,270,0,OK,0.0,527.0,15.0,0.0,1.0,15.0,38.0,2846.0,0.0
xh,3090,3090,579,0,OK,0.0,1611.0,40.0,0.0,0.0,140.0,260.0,1050.0,0.0
yo,41803,41803,3403,0,OK,0.0,5051.0,1851.0,0.0,32.0,289.0,572.0,10907.0,0.0


In [6]:
exemples("fon")


R9 apostrophes et guillemets (1456 fois)
   avant : 'danɖozan e sin tǐnmɛ nyi «Adan ɖo zan tɔn».\nWuntun'
   après : 'danɖozan e sin tǐnmɛ nyi "Adan ɖo zan tɔn".\nWuntun'
   avant : 'ɔn lɛ mɛ ɔ, é na xá:\nIl n’y a de richesse que d’ho'
   après : "ɔn lɛ mɛ ɔ, é na xá:\nIl n'y a de richesse que d'ho"
   avant : 'ísɔ́ sín nuwiwa è azɔxwé «INSAE» tɔn bló ɖo xwè 20'
   après : 'ísɔ́ sín nuwiwa è azɔxwé "INSAE" tɔn bló ɖo xwè 20'

R2 espaces spéciales (450 fois)
   avant : 'ntín ɔ̀ ɖadonú ɖe wɛ bɔ è\xa0 sɔ́ ahwàkwín do blo ná.'
   après : 'ntín ɔ̀ ɖadonú ɖe wɛ bɔ è  sɔ́ ahwàkwín do blo ná.'
   avant : 'be hweyixɔwaji nu Benɛ to\xa0ɔ bo ɖo gletoxo 10. Yě ɖ'
   après : 'be hweyixɔwaji nu Benɛ to ɔ bo ɖo gletoxo 10. Yě ɖ'
   avant : 'ukun ɖevo lɛɛ. E nɔ ba ɖɔ\xa0 e ni nyi sɛn ayikungban'
   après : 'ukun ɖevo lɛɛ. E nɔ ba ɖɔ  e ni nyi sɛn ayikungban'

R8 espaces en trop (209 fois)
   avant : 'tín ɔ̀ ɖadonú ɖe wɛ bɔ è  sɔ́ ahwàkwín do blo ná. '
   après : 'tín ɔ̀ ɖadonú ɖe wɛ bɔ è sɔ́ ahwàkwín